In [ ]:
# =========================================
# Phase 4 - Activity Detection (Onset / Offset) - Full version with EMG fallback
# Last update: 2026-02-08 (improved fallback for ALS cases)
# =========================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─── Paths ────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

PHASE2_DIR   = PROCESSED_ROOT / "phase_02_preprocess_emg"
PHASE2B_DIR  = PROCESSED_ROOT / "phase_02b_preprocess_imu"
PHASE3_DIR   = PROCESSED_ROOT / "phase_03_normalization"
PHASE3_NORM_DIR = PHASE3_DIR / "normalized_env_per_subject"  # fixed: exact path for env_norm
PHASE4_DIR   = PROCESSED_ROOT / "phase_04_events"

PHASE4_DIR.mkdir(parents=True, exist_ok=True)

print("Paths and initial settings loaded.")

# ─── Task-specific settings ─────────────────────────
TASK_SETTINGS = {
    "lifting":    {"min_duration_s": 1.8, "merge_gap_s": 0.35, "min_peak_gate": 100.0},
    "drinking":   {"min_duration_s": 1.2, "merge_gap_s": 0.30, "min_peak_gate": 60.0},
    "pick&place": {"min_duration_s": 2.2, "merge_gap_s": 1.20, "min_peak_gate": 75.0},
    "default":    {"min_duration_s": 1.5, "merge_gap_s": 0.40, "min_peak_gate": 70.0}
}

# Settings for EMG fallback mode (normalized envelope)
EMG_FALLBACK_SETTINGS = {
    "k_on": 2.0,                    # lower for better detection of weak ALS bursts
    "k_off": 1.0,                   # much lower → longer episode retention
    "peak_multiplier": 1.2,         # accept smaller peaks
    "min_duration_factor": 0.6,     # reduces min_duration_s by 40%
    "merge_gap_factor": 2.0,        # doubles merge_gap_s
    "secondary_merge_gap": 3.5      # much stronger merging to avoid fragmentation
}
# Minimum episode duration to keep (in seconds) - shorter episodes will be discarded as non-realistic
MIN_VALID_EPISODE_DURATION = 3.50  # Adjust based on visual inspection (e.g. 0.5 to 1.0 s)
MIN_VALID_PEAK_GATE = 50.0
# ─── Episode detection with hysteresis and post-processing ────────
def detect_episodes_hysteresis(
    t: np.ndarray,
    gate: np.ndarray,
    method: str = "mad",
    k_on: float = 5.0,
    k_off: float = 3.5,
    min_duration_s: float = 1.8,
    merge_gap_s: float = 0.45,
    pad_s: float = 0.15,
    peak_multiplier: float = 2.5,
    secondary_merge_gap: float = 0.8,
) -> tuple[pd.DataFrame, dict]:
    
    t   = np.asarray(t, dtype=float)
    gate = np.asarray(gate, dtype=float)

    mask_valid = np.isfinite(t) & np.isfinite(gate)
    t   = t[mask_valid]
    gate = gate[mask_valid]

    log = {
        "raw_samples": len(t),
        "gate_median": float(np.median(gate)) if len(gate) > 0 else np.nan,
        "gate_95p": float(np.percentile(gate, 95)) if len(gate) > 0 else np.nan,
    }

    if len(t) < 20:
        log["warning"] = "too_few_samples"
        return pd.DataFrame(columns=["t_on","t_off","duration_s","thr_on","thr_off","peak_gate"]), log

    med = float(np.median(gate))
    mad = float(np.median(np.abs(gate - med))) + 1e-12
    thr_on  = med + k_on  * mad
    thr_off = med + k_off * mad

    log["thr_on"]  = float(thr_on)
    log["thr_off"] = float(thr_off)

    # Fix: if thr_on too high (above 95p), automatically reduce k
    if thr_on > log["gate_95p"]:
        k_on = max(1.0, k_on * 0.7)  # reduce by 30%
        thr_on = med + k_on * mad
        log["thr_on_adjusted"] = thr_on
        log["warning"] = "thr_adjusted_down"

    episodes = []
    in_episode = False
    t_start = None

    for i in range(len(t)):
        if not in_episode:
            if gate[i] >= thr_on:
                in_episode = True
                t_start = float(t[i])
        else:
            if gate[i] <= thr_off:
                episodes.append([t_start, float(t[i])])
                in_episode = False
                t_start = None

    if in_episode:
        episodes.append([t_start, float(t[-1])])
        log["last_episode_open"] = True

    if not episodes:
        log["warning"] = "no_episodes_detected"
        return pd.DataFrame(columns=["t_on","t_off","duration_s","thr_on","thr_off","peak_gate"]), log

    t_min, t_max = float(t[0]), float(t[-1])
    padded = [[max(t_min, a - pad_s), min(t_max, b + pad_s)] for a, b in episodes]

    padded.sort(key=lambda x: x[0])
    merged = [padded[0]]
    for curr in padded[1:]:
        if curr[0] - merged[-1][1] <= merge_gap_s:
            merged[-1][1] = max(merged[-1][1], curr[1])
        else:
            merged.append(curr)

    final_merged = []
    current = merged[0]
    for nxt in merged[1:]:
        if nxt[0] - current[1] <= secondary_merge_gap:
            current[1] = max(current[1], nxt[1])
        else:
            final_merged.append(current)
            current = nxt
    final_merged.append(current)

    # Stronger merging for close intervals (< 1.2 s gap)
    secondary_merged = []
    current = final_merged[0] if final_merged else None
    for nxt in final_merged[1:]:
        if nxt[0] - current[1] <= 1.2:
            current[1] = max(current[1], nxt[1])
        else:
            secondary_merged.append(current)
            current = nxt
    if current:
        secondary_merged.append(current)

    final_merged = secondary_merged

    adaptive_peak_thr = med + peak_multiplier * mad
    log["adaptive_peak_thr"] = adaptive_peak_thr

    final_ep = []
    for a, b in final_merged:
        dur = b - a
        if dur < min_duration_s:
            continue

        segment_gate = gate[(t >= a) & (t <= b)]
        if len(segment_gate) == 0:
            continue
        peak = float(np.max(segment_gate))

        if peak >= adaptive_peak_thr:
            final_ep.append([a, b, dur, thr_on, thr_off, peak])

    df_ep = pd.DataFrame(final_ep, columns=["t_on", "t_off", "duration_s", "thr_on", "thr_off", "peak_gate"])
    log["n_episodes_after_postproc"] = len(df_ep)

    return df_ep, log


# ─── Build gate from IMU ────────────────────────────────────────
def build_gate(
    df_imu_trial: pd.DataFrame,
    use_acc_gate: bool = True,
    alpha: float = 1.0,
    time_bin_res: float = 0.001,
) -> tuple:
    df = df_imu_trial.copy()

    df["t_bin"] = (df["t_imu"] / time_bin_res).round().astype(int)

    agg_dict = {"t_imu": "mean", "gyro_gate": "median"}
    if use_acc_gate and "acc_gate" in df.columns:
        agg_dict["acc_gate"] = "median"

    df_agg = df.groupby("t_bin", as_index=False).agg(agg_dict).sort_values("t_imu")

    t    = df_agg["t_imu"].to_numpy(dtype=float)
    gyro = df_agg["gyro_gate"].to_numpy(dtype=float)

    if use_acc_gate and "acc_gate" in df_agg:
        acc  = df_agg["acc_gate"].to_numpy(dtype=float)
        gate = np.maximum(gyro, alpha * acc)
        source = f"max(gyro_gate, {alpha:.2f} × acc_gate)"
    else:
        gate   = gyro
        source = "gyro_gate"

    log_info = {
        "n_sensors_used": int(df["sensor"].nunique()) if "sensor" in df else 1,
        "gate_median": float(np.median(gate)),
        "gate_95p": float(np.percentile(gate, 95)),
    }

    return t, gate, source, "median_over_sensors", log_info


# ─── Build gate from EMG (fallback) ────────────────────────────────
def build_gate_from_emg(
    df_emg_trial: pd.DataFrame,
    time_bin_res: float = 0.001,
) -> tuple:
    df = df_emg_trial.copy()
    
    env_cols = [col for col in df.columns if col.startswith('env_norm_')]
    if not env_cols:
        env_cols = [col for col in df.columns if 'env_norm' in col.lower()]
    if not env_cols:
        raise ValueError("No env_norm columns found. Check column names.")
    
    print(f"  env_norm columns found for fallback: {env_cols}")
    
    # Important fix: use max instead of mean → more sensitive to strongest muscle activity
    df["emg_gate"] = df[env_cols].max(axis=1)
    
    df["t_bin"] = (df["t_emg"] / time_bin_res).round().astype(int)
    df_agg = df.groupby("t_bin", as_index=False).agg(
        {"t_emg": "mean", "emg_gate": "median"}
    ).sort_values("t_emg")
    
    t    = df_agg["t_emg"].to_numpy(dtype=float)
    gate = df_agg["emg_gate"].to_numpy(dtype=float)
    
    # If variance is very low, boost gate
    gate_var = np.var(gate)
    if gate_var < 1e-5:
        gate = gate * 5.0  # adjust factor as needed
        print("  Warning: gate variance too low – scaled gate")
    
    source = f"max_of_{len(env_cols)}_env_norm_muscles"
    
    log_info = {
        "n_muscles_used": len(env_cols),
        "gate_median": float(np.median(gate)),
        "gate_95p": float(np.percentile(gate, 95)),
        "gate_var": gate_var,
    }
    
    return t, gate, source, "median_over_time_bins", log_info


# ─── Plot gate and episodes ──────────────────────────────────────
def plot_trial_with_episodes(subject_name, trial_id, episodes_df):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"  # fixed: correct path
    
    ep = episodes_df[
        (episodes_df["subject"] == subject_name) &
        (episodes_df["trial_id"] == trial_id) &
        (episodes_df["duration_s"] >= MIN_VALID_EPISODE_DURATION)  # Only show valid episodes
    ].copy().sort_values("t_on")
    
    plt.figure(figsize=(14, 5))
    
    gate_source = "Unknown"
    
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_trial = df_imu[(df_imu["subject"] == subject_name) & (df_imu["trial_id"] == trial_id)]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_imu")
            t, gate, gate_source, _, _ = build_gate(df_trial, use_acc_gate=True, alpha=1.0)
            plt.plot(t, gate, lw=1.3, label=f"gate: {gate_source} (IMU)")
    
    if gate_source == "Unknown" and emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_trial = df_emg[(df_emg["subject"] == subject_name) & (df_emg["trial_id"] == trial_id)]
        if not df_trial.empty:
            df_trial = df_trial.sort_values("t_emg")
            t, gate, gate_source, _, _ = build_gate_from_emg(df_trial)
            plt.plot(t, gate, lw=1.3, label=f"gate: {gate_source} (EMG fallback)")
    
    if gate_source == "Unknown":
        print(f"No valid gate built for {trial_id} (neither IMU nor EMG)")
        plt.close()
        return
    
    for i, (_, r) in enumerate(ep.iterrows()):
        plt.axvspan(r["t_on"], r["t_off"], alpha=0.18, color='orange')
        plt.axvline(r["t_on"],  ls="--", lw=1.1, color='darkgreen',  label='on'  if i==0 else None)
        plt.axvline(r["t_off"], ls="--", lw=1.1, color='darkred',    label='off' if i==0 else None)
    
    plt.title(f"{subject_name} — {trial_id} — Episodes: {len(ep)}")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Gate")
    plt.grid(True, alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.show()


# ─── Main run for one subject ────────────────────────────────
def run_phase4_for_subject(
    subject_name: str,
    method: str = "mad",
    k_on: float = 5.0,
    k_off: float = 3.50,
    pad_s: float = 0.15,
    use_acc_gate: bool = True,
    alpha_acc: float = 1.0,
    compute_sanity_check: bool = True,
):
    imu_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    emg_path = PHASE3_NORM_DIR / f"{subject_name}__emg_env_norm.parquet"  # fixed: correct path
    
    # Load data
    df_imu = pd.DataFrame()
    df_emg = pd.DataFrame()
    
    if imu_path.exists():
        df_imu = pd.read_parquet(imu_path)
        df_imu["subject"] = df_imu["subject"].astype(str)
        df_imu["trial_id"] = df_imu["trial_id"].astype(str)
    
    if emg_path.exists():
        df_emg = pd.read_parquet(emg_path)
        df_emg["subject"] = df_emg["subject"].astype(str)
        df_emg["trial_id"] = df_emg["trial_id"].astype(str)
    
    if df_imu.empty and df_emg.empty:
        raise FileNotFoundError(f"No data found for {subject_name} (neither IMU nor EMG)")
    
    # Union of all trials
    trials_imu = set(df_imu["trial_id"].unique()) if not df_imu.empty else set()
    trials_emg = set(df_emg["trial_id"].unique()) if not df_emg.empty else set()
    all_trials = sorted(trials_imu | trials_emg)
    
    print(f"Total unique trials: {len(all_trials)}")
    print(f"  • With IMU:      {len(trials_imu)}")
    print(f"  • EMG only:      {len(trials_emg - trials_imu)}")
    
    events = []
    logs = []

    for trial_id in all_trials:
        use_imu = trial_id in trials_imu
        
        if use_imu:
            df_trial = df_imu[df_imu["trial_id"] == trial_id].copy()
            source_type = "IMU"
        else:
            df_trial = df_emg[df_emg["trial_id"] == trial_id].copy()
            source_type = "EMG_fallback"
        
        if df_trial.empty:
            print(f"Warning: trial {trial_id} is empty → skipped")
            continue
        
        print(f"Processing {trial_id:30} → source: {source_type}")
        
        task_key = next((k for k in TASK_SETTINGS if k in trial_id.lower()), "default")
        params = TASK_SETTINGS[task_key].copy()
        
        if not use_imu:
            params["min_duration_s"] *= EMG_FALLBACK_SETTINGS["min_duration_factor"]  # reduce min_duration for fallback
        
        if use_imu:
            t, gate, gate_source, _, gate_log = build_gate(
                df_trial, use_acc_gate=use_acc_gate, alpha=alpha_acc
            )
            curr_k_on  = k_on
            curr_k_off = k_off
            curr_peak_mult = 2.5
            curr_min_dur = params["min_duration_s"]
            curr_merge_gap = params["merge_gap_s"]
            curr_sec_merge = 0.8
            gate_log["data_source"] = "imu"
        else:
            t, gate, gate_source, _, gate_log = build_gate_from_emg(df_trial)
            curr_k_on  = EMG_FALLBACK_SETTINGS["k_on"]
            curr_k_off = EMG_FALLBACK_SETTINGS["k_off"]
            curr_peak_mult = EMG_FALLBACK_SETTINGS["peak_multiplier"]
            curr_min_dur = params["min_duration_s"] * EMG_FALLBACK_SETTINGS["min_duration_factor"]
            curr_merge_gap = params["merge_gap_s"] * EMG_FALLBACK_SETTINGS["merge_gap_factor"]
            curr_sec_merge = EMG_FALLBACK_SETTINGS["secondary_merge_gap"]
            gate_log["data_source"] = "emg_only"

        # Gate quality check
        if compute_sanity_check:
            gate_var = np.var(gate)
            gate_log["gate_var"] = gate_var
            if gate_var < 1e-5:
                gate_log["sanity_flag"] = "very_low_variance"
                print(f"  → Warning: gate variance very low ({gate_var:.8f})")

        ep_df, ep_log = detect_episodes_hysteresis(
            t, gate,
            method=method,
            k_on=curr_k_on,
            k_off=curr_k_off,
            min_duration_s=curr_min_dur,
            merge_gap_s=curr_merge_gap,
            pad_s=pad_s,
            peak_multiplier=curr_peak_mult,
            secondary_merge_gap=curr_sec_merge,
        )
        # New: Filter out very short / unrealistic episodes
        if not ep_df.empty:
            before_count = len(ep_df)
            ep_df = ep_df[ep_df["duration_s"] >= MIN_VALID_EPISODE_DURATION].copy()
            discarded = before_count - len(ep_df)
            if discarded > 0:
                print(f"  → Discarded {discarded} short episodes (< {MIN_VALID_EPISODE_DURATION}s)")
                ep_log["short_episodes_discarded"] = discarded
                ep_log["min_valid_duration"] = MIN_VALID_EPISODE_DURATION
        
        log = {
            "subject": subject_name,
            "trial_id": trial_id,
            "task_type": task_key,
            "gate_source": gate_source,
            "data_source": source_type,
            "method": method,
            **gate_log,
            **ep_log,
        }
        
        if ep_df.empty:
            print(f"  → Warning: no episodes detected for {trial_id} – log: {ep_log.get('warning', 'unknown')}")
            logs.append(log)
            continue

        
        ep_df["peak_gate"] = [
            float(np.nanmax(gate[(t >= a) & (t <= b)])) if np.any((t >= a) & (t <= b)) else np.nan
            for a, b in zip(ep_df["t_on"], ep_df["t_off"])
        ]
        
        ep_df = ep_df.sort_values("t_on").reset_index(drop=True)
        ep_df["episode_id"] = range(1, len(ep_df) + 1)
        
        for _, row in ep_df.iterrows():
            r = {
                "subject": subject_name,
                "trial_id": trial_id,
                "episode_id": int(row["episode_id"]),
                "t_on": float(row["t_on"]),
                "t_off": float(row["t_off"]),
                "duration_s": float(row["duration_s"]),
                "peak_gate": row["peak_gate"],
                "thr_on": float(row["thr_on"]),
                "thr_off": float(row["thr_off"]),
                "gate_source": gate_source,
                "task_type": task_key,
                "data_source": source_type,
            }
            events.append(r)
        
        logs.append(log)
    
    events_df = pd.DataFrame(events)
    logs_df   = pd.DataFrame(logs)
    
    # Save
    events_df.to_parquet(PHASE4_DIR / f"{subject_name}__episodes.parquet", index=False)
    logs_df.to_csv(  PHASE4_DIR / f"{subject_name}__episodes_log.csv", index=False)
    
    print(f"\nProcessing {subject_name} completed.")
    print(f"   → episodes: {PHASE4_DIR / f'{subject_name}__episodes.parquet'}")
    print(f"   → log:      {PHASE4_DIR / f'{subject_name}__episodes_log.csv'}")
    
    return events_df, logs_df

In [ ]:
# ─── Run cell ────────────────────────────────────────────────
SUBJECT_NAME = "ALS_Subject_3"   # <- change only this line

print(f"\n=== Starting Phase 4 processing for {SUBJECT_NAME} ===\n")
print(f"Current filter settings: min_duration={MIN_VALID_EPISODE_DURATION}s | min_peak_gate={MIN_VALID_PEAK_GATE}")

events_df, logs_df = run_phase4_for_subject(
    subject_name = SUBJECT_NAME,
    method       = "mad",
    k_on         = 5.0,
    k_off        = 3.50,
    pad_s        = 0.15,
    use_acc_gate = True,
    alpha_acc    = 1.0,
    compute_sanity_check = True,
)

# ─── Display summary results ────────────────────────────────
if not events_df.empty:
    print("\nEpisode duration distribution (seconds):")
    display(events_df["duration_s"].describe())
    
    print("\nNumber of episodes per trial and data source:")
    display(events_df.groupby(["trial_id", "data_source"])["episode_id"].count().unstack(fill_value=0))
    
    # discarded summary (if present in logs_df)
    if 'short_episodes_discarded' in logs_df.columns:
        total_discarded = logs_df["short_episodes_discarded"].sum()
        print(f"\nTotal discarded short episodes across all trials: {total_discarded}")
        display(logs_df[logs_df["short_episodes_discarded"] > 0][["trial_id", "short_episodes_discarded", "task_type"]])
    
    print(f"\nPlotting {len(events_df['trial_id'].unique())} valid trials (after filtering):")
    for tid in sorted(events_df["trial_id"].unique()):
        print(f"  → {tid}")
        plot_trial_with_episodes(SUBJECT_NAME, tid, events_df)  # without source_type - compatible with the current notebook
    
else:
    print("No episodes detected after filtering → check logs.")
    display(logs_df)